In [ ]:
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from process_census_data import prepare_census_data

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
train_x, val_x, test_x, train_y, val_y, test_y = prepare_census_data(
    random_state=99,
    split_validation=True,
    rebalance=True,
 )

train_x.head()

Get the data in tensor format

In [ ]:
train_x_tensor = torch.tensor(train_x.to_numpy(), dtype=torch.float32)
val_x_tensor = torch.tensor(val_x.to_numpy(), dtype=torch.float32)
test_x_tensor = torch.tensor(test_x.to_numpy(), dtype=torch.float32)
train_y_tensor = torch.tensor(train_y.to_numpy(), dtype=torch.float32).unsqueeze(1)
val_y_tensor = torch.tensor(val_y.to_numpy(), dtype=torch.float32).unsqueeze(1)
test_y_tensor = torch.tensor(test_y.to_numpy(), dtype=torch.float32).unsqueeze(1)

Normalize the data

In [ ]:
train_mean = train_x_tensor.mean(dim=0, keepdim=True)
train_std = train_x_tensor.std(dim=0, keepdim=True)
train_std = torch.where(train_std == 0, torch.ones_like(train_std), train_std)

train_x_tensor = (train_x_tensor - train_mean) / train_std
val_x_tensor = (val_x_tensor - train_mean) / train_std
test_x_tensor = (test_x_tensor - train_mean) / train_std

Make the dataloaders

In [ ]:
train_loader = DataLoader(
    TensorDataset(train_x_tensor, train_y_tensor),
    batch_size=512,
    shuffle=True,
 )
val_loader = DataLoader(TensorDataset(val_x_tensor, val_y_tensor), batch_size=1024)
test_loader = DataLoader(TensorDataset(test_x_tensor, test_y_tensor), batch_size=1024)

Define a simple network

In [ ]:
class BasicNN(nn.Module):
    def __init__(self, input_size, num_classes):
        super().__init__()

        self.input_size = input_size
        self.num_classes = num_classes

        self.linear_1 = nn.Linear(input_size, 5)
        self.linear_2 = nn.Linear(5, num_classes)

    def forward(self, x):
        x = nn.ReLU()(self.linear_1(x))
        x = nn.Sigmoid()(self.linear_2(x))

        return x

Inspect the model

After defining a model, it is useful to print its layers, inspect parameter shapes, and count how many trainable parameters it has.

In [ ]:
inspection_model = BasicNN(input_size=train_x_tensor.shape[1], num_classes=1)
print(inspection_model)

for name, parameter in inspection_model.named_parameters():
    print(f"{name}: shape={tuple(parameter.shape)}, trainable={parameter.requires_grad}")

sum(parameter.numel() for parameter in inspection_model.parameters() if parameter.requires_grad)

Define an evaluation function

In [ ]:
def evaluate_model(model, data_loader, device, threshold=0.5):
    model.eval()
    all_targets = []
    all_predictions = []

    with torch.no_grad():
        for features, targets in data_loader:
            features = features.to(device)
            predictions = model(features)
            all_predictions.append(predictions)
            all_targets.append(targets)

    y_true = torch.cat(all_targets).squeeze(1).numpy()
    y_pred = torch.cat(all_predictions).squeeze(1).numpy()
    y_pred = (y_pred >= threshold).astype(int)

    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
    }

Define the training function

In [ ]:
def train_model(model, train_loader, val_loader, device, epochs=12, learning_rate=1e-3):
    torch.manual_seed(99)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    loss_fn = nn.BCEWithLogitsLoss()

    history = []
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0

        for features, targets in train_loader:
            features = features.to(device)
            targets = targets.to(device)

            optimizer.zero_grad()
            logits = model(features)
            loss = loss_fn(logits, targets)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * features.size(0)

        validation_metrics = evaluate_model(model, val_loader, device)
        history.append(
            {
                "epoch": epoch + 1,
                "train_loss": running_loss / len(train_loader.dataset),
                **validation_metrics,
            }
        )

    return model, pd.DataFrame(history)

Train the model

In [ ]:
model = BasicNN(
        input_size=126,
        num_classes=1
    ).to(device)
trained_model, training_history = train_model(
    model, train_loader, val_loader, device, epochs=10, learning_rate=2e-3
)

In [ ]:
ax = training_history.plot(x="epoch", y="train_loss", figsize=(12, 8), color="tab:blue")
training_history.plot(
    x="epoch",
    y="recall",
    secondary_y=True,
    ax=ax,
    color="tab:orange",
)

ax.set_ylabel("train_loss")
ax.right_ax.set_ylabel("recall");

# Exercise: Discover Neural Network Architectures

We have a very basic neural network to determine if someone is earning more than $50k, but we can improve the performance of the basic model by change the architecture of the model. Your goal is to improve the predictive performance of this model by experimenting with the network architecture. You are free to modify the model, but keep the data used in the model the same.

### What you can change
Try different architectural choices such as:
- Depth; Adding or removing layers
- Width; Increase or decrease the number of neurons per layer
- Activation functions; Replace ReLU with alternatives (e.g., LeakyReLU, Tanh)
- Regularization: Add dropout layers, use batch normalization
- Output layer behavior; How do we convert the result to 0, 1

When adjusting the network, always also note down how many parameters the network has. Train and evaluate on the same dataset split. As the dataset is very unbalanced we are most interested in having a good recall. Track your results and identify which architectural choice led to improvements.

### Questions to answer
- Which architectural changes had the largest positive impact on performance?
- Did increasing model complexity always improve performance?
- Did you observe signs of overfitting? How did you address them?
- How does your improved model compare to:
- - The baseline model
- - The 10-feature model from the XAI exercise

### Optional bonus (advanced)

- Perform a systematic search (e.g., small grid search over depth/width)
- Compare with a tree-based model (e.g., XGBoost) from the previous exercises
- Reflect on whether the more complex neural network is actually worth it 